# 03: Algorithms by hand: distances, clusters, trees, projections and weights

**Practice set:** `practice/03-algorithms-by-hand.md` · **Exam task:** Jan 18, Section B4 (Model Compression), pages 23–26 · **Moves:** M1, M6, M10

## Hook

You strap an accelerometer to your wrist and record **20 labelled gestures** for your robot: 10 *waves* and 10 *punches*. Each gesture becomes two numbers: how hard it peaks, and how long it lasts. The microcontroller must now decide, for every new gesture, which one it was.

Your first try is the simplest rule there is: find the stored gesture closest to the new one and copy its label. It works, until a gesture arrives that is *exactly* as close to a wave as to a punch. What should the robot do? Would the answer change if you measured "close" differently?

You have other options too. You could let the 3 nearest gestures vote. You could ask two yes/no questions about the numbers (a decision tree), squash the two numbers into one before deciding (PCA), or check the sign of a weighted sum (a linear classifier). And if you hadn't labelled anything, could the robot still find the two kinds of gesture by itself (k-means, clustering)?

Each of these decides by a rule simple enough to run by hand. That's exactly what the exam tests: it gives you the rule and expects you to execute it **exactly**, ties and edge cases included. One tie broken the wrong way flips the answer.

## Before you start

- **You need:** coordinates, Pythagoras' theorem and averages. Section 7 uses a quadratic equation; the formula is given there. Problem 1.4 from notebook 01 (mean vs median) helps in section 3.
- **How to use this notebook:** run the cells in order. When you see **PREDICT**, write your guess *before* running the next cell. When you see **BUILD**, write the function yourself; the test cell tells you when it's right. Hints sit in collapsed blocks. Open one only after a real attempt.
- **Read the rule you're given** (move M1 from notebook 01). Every exam question defines its algorithm, including how to break ties. When this notebook states a tie rule, it's an example; on the exam, use theirs.
- **Pen and paper stay on the desk.** The exam has no computer. Use the notebook to check and explore *after* you've reasoned it out.

In [ ]:
import prep
import math
import itertools
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

---
## 1. Distance: two different rulers

**Idea.** k-NN, k-means and clustering all need an answer to "how far apart are these two points?". Two common rulers:
- **Euclidean distance**: the straight-line distance, from Pythagoras: √(Δx² + Δy²). Here Δx is the difference in x, and Δy the difference in y.
- **Manhattan distance**: |Δx| + |Δy|. (|·| is the **absolute value**: drop the minus sign, so |−3| = 3.) It's the distance if you can only move along the axes, like a robot on a grid floor, or a taxi in Manhattan's street grid.

**Smallest example.** Distances from the origin (0, 0) to four points.

In [ ]:
pts = [(0, 5), (3, 3), (4, -1), (-2, 0)]
rows = []
for x, y in pts:
    rows.append([(x, y), x, y, math.sqrt(x * x + y * y), abs(x) + abs(y)])
prep.table(rows, header=["point", "Δx", "Δy", "Euclidean", "Manhattan"])

**PREDICT:**
1. Rank the four points from nearest to farthest, once with each ruler. Does the order change?
2. On paper, sketch every point at Euclidean distance exactly 1 from the origin. That's a circle. Now sketch every point at Manhattan distance exactly 1. What shape is it?

In [ ]:
theta = np.linspace(0, 2 * np.pi, 400)
plt.figure(figsize=(4, 4.6))
plt.plot(np.cos(theta), np.sin(theta), label="Euclidean distance = 1")
plt.plot([1, 0, -1, 0, 1], [0, 1, 0, -1, 0], label="Manhattan distance = 1")
plt.scatter([0.707], [0.707], color="k", zorder=3)
plt.annotate("(0.707, 0.707)", (0.707, 0.707), textcoords="offset points", xytext=(5, 5))
plt.gca().set_aspect("equal"); plt.grid(alpha=.3)
plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), fontsize=8, ncol=1)
plt.title("'Unit circles' of the two rulers")
plt.show()

The Manhattan "circle" is a diamond. On the axes the two rulers agree. Along a diagonal they don't: the point (0.707, 0.707) is at Euclidean distance 1 but Manhattan distance 1.414. **Diagonal neighbours look farther under Manhattan.** So switching rulers can reorder who is nearest, and that can change a prediction. The ruler is part of the model, not a detail.

### BUILD 1: `dist`
`metric` is `"euclid"` or `"manhattan"`.

In [ ]:
def dist(p, q, metric):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(dist, [
    (((0, 0), (3, 4), "euclid"), 5.0),
    (((0, 0), (3, 4), "manhattan"), 7),
    (((1, 1), (1, 1), "euclid"), 0.0),
    (((-1, 2), (2, -2), "manhattan"), 7),
    (((-1, 2), (2, -2), "euclid"), 5.0),
])

---
## 2. k-NN, and when a tie matters

**Idea.** To classify a new point (the **query**), find the k labelled points nearest to it (its **k nearest neighbours**) and take a **majority vote** of their labels. There is no training at all: the stored data *is* the model.

Ties come in two kinds:
1. **Distance ties**: several points are equally far away, competing for the last places in the top k. Which ones get in?
2. **Vote ties**: two labels get the same number of votes.

The exam either gives you a tie rule or asks whether the tie matters. The move for the second kind of question: **try every way of breaking the tie. If they all give the same label, the tie doesn't matter.**

**Smallest example.** Query (0, 0), k = 3, Euclidean distance.

In [ ]:
kpts = [(1, 0), (0, 2), (-2, 0), (0, -2), (3, 3)]
klabels = ["wave", "punch", "wave", "punch", "punch"]
order = sorted(range(len(kpts)), key=lambda i: kpts[i][0] ** 2 + kpts[i][1] ** 2)
prep.table([[kpts[i], klabels[i], kpts[i][0] ** 2 + kpts[i][1] ** 2, math.sqrt(kpts[i][0] ** 2 + kpts[i][1] ** 2)]
            for i in order], header=["point", "label", "squared distance", "distance"])

**PREDICT:** which points are *certainly* among the 3 nearest? Which are tied for the remaining places? Does the tie change the vote?

In [ ]:
must = [i for i in range(5) if kpts[i] == (1, 0)]
tied = [1, 2, 3]   # the three points at distance 2
for extra in itertools.combinations(tied, 2):
    chosen = must + list(extra)
    votes = Counter(klabels[i] for i in chosen)
    print("neighbours", [kpts[i] for i in chosen], " votes", dict(votes), " ->", votes.most_common(1)[0][0])

Here the tie **does** matter: depending on which two of the three tied points get in, the answer is wave or punch. A robot needs a rule for this. Common rules are "include every tied point", "take points in the order they were stored", or "fall back to k = 1". Notice the vote can't tie at k = 3 with two labels: an odd k avoids vote ties, but not distance ties.

**Tip:** compare **squared** distances. Squaring never changes the order of non-negative numbers (if a > b ≥ 0 then a² > b²), so the neighbours are the same. You avoid square roots, and with whole-number coordinates the ties are exact. This is the same reason AUC in notebook 01 didn't change when scores were squared.

### BUILD 2: `knn_predict`
Return the **set** of labels that could win, over every way of breaking a distance tie for the last places. If the vote itself ties, include every tied label. So `{"wave"}` means the answer is certain, and `{"wave", "punch"}` means the tie matters.

<details><summary>Hint</summary>

Find d_k, the k-th smallest distance. Points closer than d_k are certainly in. Points at exactly d_k are tied, and you need k − (number certainly in) of them. `itertools.combinations(tied, m)` lists every way to choose m of them.

</details>

In [ ]:
def knn_predict(points, labels, query, k, metric):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(knn_predict, [
    ((kpts, klabels, (0, 0), 3, "euclid"), {"wave", "punch"}),
    ((kpts, klabels, (0, 0), 1, "euclid"), {"wave"}),
    ((kpts, klabels, (0, 0), 5, "euclid"), {"punch"}),
    ((kpts, klabels, (0, 0), 4, "manhattan"), {"wave", "punch"}),
    (([(0, 5), (3, 3)], ["x", "y"], (0, 0), 1, "manhattan"), {"x"}),
    (([(0, 5), (3, 3)], ["x", "y"], (0, 0), 1, "euclid"), {"y"}),
])

### EXPLORE: what does k-NN do with your 20 gestures?
Here are the hook's 20 gestures (x = peak acceleration, y = duration, both on a 0–10 scale). The plot colours every possible new gesture by its 3-NN prediction. Pale cells are places where a tie decides the answer.

**PREDICT:** do the two rulers draw the same boundary? Where do you expect the pale "tie matters" cells?

In [ ]:
waves = [(1, 7), (2, 8), (2, 6), (3, 9), (3, 7), (4, 8), (1, 5), (4, 6), (5, 9), (3, 5)]
punches = [(6, 2), (7, 3), (5, 3), (8, 1), (6, 4), (9, 2), (7, 5), (4, 2), (8, 4), (5, 1)]
if prep.ready(knn_predict, [(0, 0)], ["a"], (1, 1), 1, "euclid"):
    gpts, glabs = waves + punches, ["wave"] * 10 + ["punch"] * 10
    grid = np.arange(0, 10.01, 0.25)
    fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
    for ax, metric in zip(axes, ["euclid", "manhattan"]):
        img = np.zeros((len(grid), len(grid)))
        for i, yv in enumerate(grid):
            for j, xv in enumerate(grid):
                out = knn_predict(gpts, glabs, (xv, yv), 3, metric)
                img[i, j] = 0.5 if len(out) > 1 else (1.0 if "wave" in out else 0.0)
        ax.imshow(img, origin="lower", extent=(-0.125, 10.125, -0.125, 10.125), cmap="coolwarm", vmin=0, vmax=1, alpha=0.6)
        ax.scatter(*zip(*waves), c="darkred", marker="o", s=18, label="wave")
        ax.scatter(*zip(*punches), c="navy", marker="x", s=18, label="punch")
        ax.set_title(f"3-NN, {metric}"); ax.set_xlabel("peak"); ax.set_ylabel("duration")
    axes[0].legend(fontsize=7, loc="lower left")
    plt.tight_layout(); plt.show()

Look at the big pale corners in the Manhattan panel. Far above and to the right of every stored point, the Manhattan distance to a point (xᵢ, yᵢ) is (q_x − xᵢ) + (q_y − yᵢ) = (q_x + q_y) − (xᵢ + yᵢ). So the ranking depends only on xᵢ + yᵢ, the same everywhere in that corner. Any stored points with equal xᵢ + yᵢ stay tied across the whole region. With whole-number data, ties aren't rare accidents. Plan for them.

**Now do practice problem 3.1 on paper**, then check it. Build the full distance table first (both rulers), then vote.

In [ ]:
prep.check("3.1a", ...)   # label with Euclidean distance
prep.check("3.1b", ...)   # label with Manhattan distance

**3.1, is any tie a problem?** *Your explanation here.*

---
## 3. k-means: assign, average, repeat

**Idea.** Now you have no labels, only points, and you want k groups. k-means keeps k **centroids** (a centroid is the mean, the average position, of a group's points) and repeats two steps:
1. **Assign:** each point joins its nearest centroid.
2. **Average:** move each centroid to the mean of the points that joined it.

Stop when an assign step changes nothing. To measure how tight the clusters are, use **SSE** (sum of squared errors): the sum, over all points, of (point − its centroid)². Smaller is tighter.

Tie rule for this notebook: a point exactly halfway between two centroids joins the **left** one (the one that comes first in the list). Practice problem 3.2 uses the same rule.

**Smallest example.** Points 0, 2, 4, 9, 11, 13 with k = 2, starting from centroids 0 and 2.

**PREDICT:** do iteration 1 on paper. Then guess where the centroids end up, and after how many iterations.

In [ ]:
xs = [0, 2, 4, 9, 11, 13]
cents = [0.0, 2.0]
prev, it, rows = None, 0, []
while True:
    it += 1
    groups = [[], []]
    for x in xs:
        d0, d1 = abs(x - cents[0]), abs(x - cents[1])
        groups[0 if d0 <= d1 else 1].append(x)
    sse = sum((x - c) ** 2 for g, c in zip(groups, cents) for x in g)
    new = [sum(g) / len(g) if g else c for g, c in zip(groups, cents)]
    rows.append([it, cents, groups[0], groups[1], sse, new])
    if groups == prev:
        break
    prev, cents = groups, new
prep.table(rows, header=["iteration", "centroids", "left cluster", "right cluster", "SSE before moving", "new centroids"])

Read the SSE column top to bottom. It never goes up. That's not luck:
- **Assign** can't raise SSE: each point switches only to a *nearer* centroid.
- **Average** can't raise SSE: for a fixed group, the mean is the single point with the smallest sum of squared distances to the group. That's exactly what you proved in notebook 01, problem 1.4.

So k-means always goes downhill and must stop. But going downhill only gets you to the bottom of *the valley you started in*. That valley might not be the deepest one.

### BUILD 3: `kmeans_1d`
Return `(centroids, sse)` at convergence. A point exactly halfway joins the centroid that comes first in the list. If a centroid gets no points, leave it where it is.

In [ ]:
def kmeans_1d(xs, init):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(kmeans_1d, [
    (([0, 2, 10, 12], [0, 2]), ([1.0, 11.0], 4.0)),
    (([1, 2, 3], [2]), ([2.0], 2.0)),
    (([0, 1, 2], [0, 2]), ([0.5, 2.0], 0.5)),
    (([0, 4], [2, 10]), ([2.0, 10.0], 8.0)),
])

### EXPLORE: local minima
Three tight clumps: 0, 1, 2 · 10, 11, 12 · 20, 21, 22, with k = 3. The obvious answer has one centroid per clump. The cell below starts k-means from **every** possible choice of 3 data points as the starting centroids (84 starts).

**PREDICT:** how many *different* final answers will there be? What do you think goes wrong when all three centroids start in the left clump?

In [ ]:
if prep.ready(kmeans_1d, [0, 1], [0, 1]):
    clumps = [0, 1, 2, 10, 11, 12, 20, 21, 22]
    results = Counter()
    for init in itertools.combinations(clumps, 3):
        c, s = kmeans_1d(clumps, list(init))
        results[(tuple(round(v, 2) for v in c), round(s, 2))] += 1
    prep.table([[list(c), s, n] for (c, s), n in sorted(results.items(), key=lambda kv: kv[0][1])],
               header=["final centroids", "SSE", "number of starts"])

A **local minimum** is an answer that no single assign or average step can improve, even though a better answer exists. In the bad runs, two centroids split one clump while the third is stuck covering two clumps. Neither step can fix that, because moving any one point or centroid makes SSE worse first. The fix in practice: run k-means from several starts and keep the run with the lowest SSE.

**Now do practice problem 3.2 on paper.** Print every iteration in a table like the one above, and apply the halfway rule exactly.

In [ ]:
prep.check("3.2a", ...)       # final centroids as a list, e.g. [left, right]
prep.check("3.2a_sse", ...)
prep.check("3.2b", ...)
prep.check("3.2b_sse", ...)
prep.check("3.2c", ...)       # "a" or "b"

---
## 4. Gini impurity: how mixed is a group?

**Idea.** A **decision tree** classifies by asking yes/no questions about the features, one after another ("is the peak above 4?"). A good question splits the samples into groups that are each **pure** (all one label). So we need a number for "how mixed is this group?". That number is called **impurity**.

**Gini impurity** is defined like this: **pick two samples from the group at random, *with replacement* (the second pick may be the same sample as the first). Gini is the probability that their labels differ.**

**Smallest example.** A group of 4 gestures: wave, wave, wave, punch. List every ordered pair of picks.

In [ ]:
group = ["wave", "wave", "wave", "punch"]
rows = []
for i, a in enumerate(group):
    rows.append([f"1st pick #{i + 1} ({a})"] + ["same" if a == b else "DIFF" for b in group])
prep.table(rows, header=["", "2nd: #1", "#2", "#3", "#4"])

**PREDICT:**
1. What fraction of the 16 pairs differ?
2. Write that probability as a formula using p_wave = 3/4 and p_punch = 1/4. (Hint: first find P(both picks are wave) and P(both picks are punch). The two picks are independent; notebook 02 tells you how to combine them.)

In [ ]:
diff = sum(a != b for a in group for b in group)
p_wave, p_punch = 3 / 4, 1 / 4
print(f"pairs that differ: {diff}/16 = {diff / 16}")
print(f"P(both wave) = {p_wave}^2 = {p_wave ** 2},  P(both punch) = {p_punch}^2 = {p_punch ** 2}")
print(f"P(same) = {p_wave ** 2 + p_punch ** 2},  P(differ) = 1 - P(same) = {1 - p_wave ** 2 - p_punch ** 2}")

**The aha: Gini = 1 − Σ pᵢ², the chance that two random picks disagree.** Σpᵢ² is the chance that both picks land in the same class (class i twice has probability pᵢ · pᵢ, then add over the classes). Gini is everything else. (Σ, the Greek letter sigma, means "add up over all the classes".)

Quick facts that follow straight from the definition:
- A pure group: the picks always agree, so Gini = 0.
- A 50/50 group of two classes: Gini = 1 − ¼ − ¼ = ½.
- Three equal classes: 1 − 3 · (1/3)² = 2/3.

### BUILD 4: `gini`
`counts` is a list of how many samples of each class are in the group.

In [ ]:
def gini(counts):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(gini, [
    (([4, 0],), 0.0),
    (([2, 2],), 0.5),
    (([3, 1],), 0.375),
    (([1, 1, 1],), 2 / 3),
    (([5],), 0.0),
])

### EXPLORE: the shape of Gini
**PREDICT:** for two classes, with a fraction p of the first class, what is the largest Gini can be, and at which p? Sketch Gini against p.

In [ ]:
if prep.ready(gini, [1, 1]):
    ps = np.linspace(0, 1, 101)
    plt.figure(figsize=(6, 3.5))
    plt.plot(ps, [gini([p, 1 - p]) for p in ps], label="Gini = 1 - p² - (1-p)²")
    plt.plot(ps, np.minimum(ps, 1 - ps), "--", label="error rate of majority vote = min(p, 1-p)")
    plt.xlabel("fraction p of class 1"); plt.ylabel("impurity"); plt.legend(fontsize=8); plt.grid(alpha=.3)
    plt.title("Impurity is highest for a 50/50 mix")
    plt.show()

Gini works out to 2p(1 − p) for two classes. It's a smooth hill, highest at a 50/50 mix. The error rate of a majority vote is a sharp tent with the same peak. Trees prefer the smooth one because it still rewards a split that makes groups purer without changing their majority label.

---
## 5. Choosing a split

**Idea.** A split sends every sample left or right. Score the split by the **size-weighted average** Gini of its two children:

**split score = (n_L ÷ n) · Gini(left) + (n_R ÷ n) · Gini(right)**

Lower is purer. **Gain** = Gini(parent) − split score, and the tree picks the split with the biggest gain (equivalently, the lowest score).

**Smallest example.** 8 gestures: 5 waves and 3 punches. Two candidate questions (counts are (waves, punches)):
- Split X: "duration above 9?" sends (1, 0) left and (4, 3) right.
- Split Y: "peak below 4?" sends (4, 1) left and (1, 2) right.

**PREDICT:** Split X makes a *perfectly pure* child. Split Y has no pure child at all. Which split should the tree choose?

In [ ]:
def g(cs):
    return 1 - sum((c / sum(cs)) ** 2 for c in cs)

parent = [5, 3]
print(f"parent {parent}: Gini = {g(parent):.4f}\n")
for name, L, R in [("X", [1, 0], [4, 3]), ("Y", [4, 1], [1, 2])]:
    n = sum(L) + sum(R)
    rows = [[side, cs, sum(cs), g(cs), sum(cs) / n, sum(cs) / n * g(cs)] for side, cs in [("left", L), ("right", R)]]
    prep.table(rows, header=["child", "counts", "size", "Gini", "weight", "weight x Gini"])
    score = rows[0][5] + rows[1][5]
    print(f"split {name}: score = {score:.4f}, gain = {g(parent) - score:.4f}   (plain unweighted average: {(rows[0][3] + rows[1][3]) / 2:.4f})\n")

**Weight by size.** Split X's pure child holds a single sample. Its other child holds 7 of the 8 samples, and that child is almost as mixed as the parent. A split is only as good as what it does to *most* of the samples. Look at the "plain unweighted average" numbers: they would pick X, which is the wrong choice. This is the trap to avoid.

### BUILD 5: `split_score`
`left` and `right` are lists of class counts. Return the size-weighted Gini of the split.

In [ ]:
def split_score(left, right):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(split_score, [
    (([4, 0], [0, 4]), 0.0),
    (([2, 2], [2, 2]), 0.5),
    (([1, 0], [4, 3]), 3 / 7),
    (([3, 0], [0, 1]), 0.0),
])

**Now do practice problem 3.3 on paper.** One of its splits has a pure child. Don't fall for it automatically: compute all three scores.

In [ ]:
prep.check("3.3", ...)        # feature letter
prep.check("3.3_gain", ...)   # floor(1000 x gain)

---
## 6. Agglomerative clustering: single vs complete linkage

**Idea.** Start with every point as its own cluster. Repeatedly **merge the two closest clusters**, and record the distance at which each merge happens (its **height**). The record of merges is drawn as a **dendrogram**, a tree diagram whose joins sit at the merge heights. **Cutting the dendrogram at height h** means keeping only the merges below h. The clusters left = (number of points) − (number of merges below h).

"Closest clusters" needs a distance between *groups* of points. Two choices:
- **Single linkage:** the distance between the *closest* pair of points, one from each cluster.
- **Complete linkage:** the distance between the *farthest* pair.

**Smallest example.** Points on a line: 0, 1, 3, 7, 12. (If two pairs of clusters were ever equally close, you'd need a tie rule. None happen here.)

**PREDICT:** do single and complete linkage merge in the same order? What is the height of the final merge under each?

In [ ]:
points_1d = [0, 1, 3, 7, 12]
for kind in ["single", "complete"]:
    clusters = [[p] for p in points_1d]
    rows = []
    while len(clusters) > 1:
        best = None
        for i in range(len(clusters)):
            for j in range(i + 1, len(clusters)):
                ds = [abs(a - b) for a in clusters[i] for b in clusters[j]]
                d = min(ds) if kind == "single" else max(ds)
                if best is None or d < best[0]:
                    best = (d, i, j)
        d, i, j = best
        rows.append([len(rows) + 1, [c for c in clusters], f"{clusters[i]} + {clusters[j]}", d])
        clusters = [c for t, c in enumerate(clusters) if t not in (i, j)] + [sorted(clusters[i] + clusters[j])]
    print(f"{kind.upper()} linkage")
    prep.table(rows, header=["merge", "clusters before", "merged", "height"])
    print()

Single linkage grows one long **chain**: 3 joins {0, 1} because 3 is close to 1, then 7 joins because it's close to 3, even though 7 is far from 0. Complete linkage asks about the *farthest* pair, so it refuses to stretch a cluster out. It prefers to pair 7 with 12 into a separate compact group. The same data gives different trees, and so a different number of clusters for the same cut. Again the definition decides the answer.

Cutting example: cut single linkage at height 3. The merges at heights 1 and 2 are below 3 and are kept, so 5 − 2 = 3 clusters remain: {0, 1, 3}, {7} and {12}.

### BUILD 6: `linkage_merges`
`points` are numbers on a line; `kind` is `"single"` or `"complete"`. Return the list of merge heights, in the order the merges happen.

In [ ]:
def linkage_merges(points, kind):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(linkage_merges, [
    (([0, 1, 5], "single"), [1, 4]),
    (([0, 1, 5], "complete"), [1, 5]),
    (([0, 10], "single"), [10]),
    (([0, 2, 3, 7], "single"), [1, 2, 4]),
    (([0, 2, 3, 7], "complete"), [1, 3, 7]),
])

### EXPLORE: clusters left at each cut height
**PREDICT:** sketch "number of clusters" against "cut height" for both linkages. Can the single-linkage curve ever be *above* the complete-linkage curve?

In [ ]:
if prep.ready(linkage_merges, [0, 1], "single"):
    H = np.linspace(0, 13, 400)
    plt.figure(figsize=(6, 3.5))
    for kind in ["single", "complete"]:
        hs = linkage_merges(points_1d, kind)
        plt.step(H, [len(points_1d) - sum(h < x for h in hs) for x in H], where="post", label=kind)
    plt.yticks(range(1, len(points_1d) + 1))
    plt.xlabel("cut height"); plt.ylabel("clusters left"); plt.legend(); plt.grid(alpha=.3)
    plt.title("Points 0, 1, 3, 7, 12")
    plt.show()

For any two clusters, the closest pair is never farther apart than the farthest pair. So single linkage always merges at a height no greater than complete linkage would, and its curve sits at or below.

**Now do practice problem 3.4 on paper.** Write a merge table like the one above for each linkage.

In [ ]:
prep.check("3.4a_single", ...)     # height of the final merge
prep.check("3.4a_complete", ...)
prep.check("3.4b_single", ...)     # clusters after cutting at height 4
prep.check("3.4b_complete", ...)

---
## 7. PCA in 2-D

Suppose the microcontroller can only afford to keep **one** number per gesture instead of two. Which single number keeps the most information? **PCA** (principal component analysis) answers that: it finds the direction along which the data is most spread out. This section builds it in four small steps.

### 7a. Shadows, and the spread along a direction
- **Centred** data: subtract the mean x from every x, and the mean y from every y. The cloud's centre is then at (0, 0).
- A **direction** at angle θ from the x-axis is the arrow u = (cos θ, sin θ). If you haven't met cos and sin: they're the coordinates of the point at angle θ on the circle of radius 1. For example θ = 0° gives (1, 0), θ = 90° gives (0, 1), and θ = 45° gives (0.707, 0.707).
- The **shadow** (projection) of a point (x, y) on the line through the origin in direction u is **t = x · cos θ + y · sin θ**. This "multiply matching parts and add" is called the **dot product** of the point with u. Example: the shadow of (2, 1) on the x-axis (θ = 0) is just 2.
- The **spread** along u is Σt², the sum of the squared shadows. (For centred data this is n × the variance of the shadows.)

**PC1**, the first principal component, is the direction with the largest spread. Keeping each point's shadow on PC1 loses the least information.

**Smallest example.** Four centred points.

In [ ]:
P = [(-2, -1), (-1, 0), (1, 1), (2, 0)]
print("mean:", (sum(p[0] for p in P) / 4, sum(p[1] for p in P) / 4))
rows, spread = [], {}
for x, y in P:
    rows.append([(x, y)] + [round(x * math.cos(math.radians(a)) + y * math.sin(math.radians(a)), 3) + 0.0 for a in (0, 45, 90)])
prep.table(rows, header=["point", "shadow at 0°", "at 45°", "at 90°"])

**PREDICT:** square and add each column. Which of 0°, 45° and 90° has the biggest spread? Is some other angle better still?

In [ ]:
def spread_at(deg):
    return sum((x * math.cos(math.radians(deg)) + y * math.sin(math.radians(deg))) ** 2 for x, y in P)

for a in (0, 45, 90):
    print(f"spread at {a:3d}°: {spread_at(a):.3f}")
angles = np.arange(0, 180, 0.5)
spreads = [spread_at(a) for a in angles]
best = angles[int(np.argmax(spreads))]
plt.figure(figsize=(6, 3.5))
plt.plot(angles, spreads)
plt.axvline(best, color="gray", ls="--")
plt.xlabel("direction θ (degrees)"); plt.ylabel("spread Σt²"); plt.grid(alpha=.3)
plt.title(f"Spread is largest near θ = {best}°")
plt.show()

None of the "obvious" angles is the best one. To find the exact best angle without sweeping, you need a little algebra.

### 7b. The scatter matrix
Expand one squared shadow: (x cos θ + y sin θ)² = x² cos²θ + 2xy sinθ cosθ + y² sin²θ. Add over all points:

**spread(θ) = cos²θ · Σx² + 2 sinθ cosθ · Σxy + sin²θ · Σy²**

So just **three sums** decide the spread in *every* direction. Arrange them in a square table of numbers, a **matrix**, called the **scatter matrix**:

S = [[Σx², Σxy], [Σxy, Σy²]], which we'll write as [[a, c], [c, b]].

**Matrix times vector**: each row of the matrix takes a dot product with the vector. So [[a, c], [c, b]] · (u₁, u₂) = (a·u₁ + c·u₂, c·u₁ + b·u₂). Tiny example: [[2, 1], [1, 3]] · (1, 0) = (2, 1). With this, spread(θ) = u · (S u). (Multiply it out and you get the formula above.)

In [ ]:
a_ = sum(x * x for x, y in P); b_ = sum(y * y for x, y in P); c_ = sum(x * y for x, y in P)
print(f"Σx² = {a_}, Σy² = {b_}, Σxy = {c_}    S = [[{a_}, {c_}], [{c_}, {b_}]]")
for deg in (0, 30, 45, 90):
    u = (math.cos(math.radians(deg)), math.sin(math.radians(deg)))
    Su = (a_ * u[0] + c_ * u[1], c_ * u[0] + b_ * u[1])
    print(f"θ = {deg:2d}°: u·(Su) = {u[0] * Su[0] + u[1] * Su[1]:.4f}   direct sum of squared shadows = {spread_at(deg):.4f}")

### 7c. Eigenvectors: directions the matrix only stretches
Multiply S by a direction u and you usually get an arrow pointing somewhere *else*: the matrix turns it as well as stretching it. Below, S is applied to arrows at several angles.

**PREDICT:** is there a direction that S doesn't turn at all? Use the spread curve from 7a to guess where it might be.

In [ ]:
rows = []
special = math.degrees(math.atan2(1, 3))   # about 18.43 degrees (you'll see where this comes from in 7d)
for deg in (0, special, 45, 90, special + 90, 135):
    u = (math.cos(math.radians(deg)), math.sin(math.radians(deg)))
    Su = (a_ * u[0] + c_ * u[1], c_ * u[0] + b_ * u[1])
    su_angle = math.degrees(math.atan2(Su[1], Su[0])) % 180
    rows.append([round(deg, 2), (round(u[0], 3), round(u[1], 3)), (round(Su[0], 3), round(Su[1], 3)),
                 round(su_angle, 2), round(math.hypot(*Su), 3)])
prep.table(rows, header=["θ of u", "u", "S u", "angle of S u", "length of S u"])

At about 18.43° the arrow comes out pointing the same way, just 11 times longer. At 108.43° (the perpendicular direction) it comes out the same way, 1 times as long. Everywhere else it gets turned.

An **eigenvector** of S is a direction that S only stretches: **S u = λ u** for some number λ. That number λ (the Greek letter lambda) is the **eigenvalue**, the stretch factor.

**The aha: PC1 is the eigenvector with the largest eigenvalue, and the spread along it equals that eigenvalue.** Compare the table with the 7a curve: the peak of the spread curve is at the "not turned" direction, and the peak height is the stretch factor.

Why must the peak be an eigenvector?
- *Spread = eigenvalue:* if S u = λ u, then spread = u · (S u) = u · (λ u) = λ, because u has length 1.
- *Peak = eigenvector* (only if you know derivatives): as θ turns, u moves in the perpendicular direction u⊥, and the slope of spread(θ) works out to 2 · (u⊥ · S u). At the peak the slope is 0, so S u has no part along u⊥. That means S u points along u.

One more fact you need: the **total spread** is Σx² + Σy², whichever pair of perpendicular directions you measure along. Each point's shadows on two perpendicular directions are just its coordinates in a rotated grid, so by Pythagoras t₁² + t₂² = x² + y². Add over the points: spread along PC1 + spread along the perpendicular direction = Σx² + Σy². So **the fraction of the spread that PC1 keeps is λ₁ ÷ (Σx² + Σy²)**. (In the table: 11 + 1 = 12 = 10 + 2.)

### 7d. Finding eigenvalues with a quadratic
S u = λ u, written out row by row:

(a − λ) u₁ + c u₂ = 0  and  c u₁ + (b − λ) u₂ = 0

Each equation is a straight line through the origin in the (u₁, u₂) plane. A non-zero u can satisfy both only if they're **the same line**, which means their coefficients are in proportion: (a − λ) ÷ c = c ÷ (b − λ). Cross-multiply:

(a − λ)(b − λ) − c² = 0, which expands to **λ² − (a + b) λ + (ab − c²) = 0**

Here a + b is called the **trace** (the sum of the diagonal), and ab − c² is the **determinant**, written det. So: **λ² − trace · λ + det = 0**. The **quadratic formula** solves it: λ = [trace ± √(trace² − 4 det)] ÷ 2.

The two roots always add up to the trace. (Multiply out (λ − λ₁)(λ − λ₂) = λ² − (λ₁ + λ₂)λ + λ₁λ₂ and compare it with the equation.) Since the trace is Σx² + Σy², the fraction PC1 keeps can also be written **λ₁ ÷ (λ₁ + λ₂)**. That's the form practice problem 3.5 uses.

Then the direction comes from the first equation: u₂ ÷ u₁ = (λ − a) ÷ c. That ratio is the slope (rise over run) of the eigenvector, and its angle is **atan(slope)**. Here atan, the inverse tangent, means "the angle whose slope is this number". For example, atan(1) = 45°.

**PREDICT:** solve the quadratic for our S = [[10, 3], [3, 2]] on paper. Then find the PC1 angle and the fraction of spread it keeps.

In [ ]:
tr, det = a_ + b_, a_ * b_ - c_ * c_
disc = tr * tr - 4 * det
l1, l2 = (tr + math.sqrt(disc)) / 2, (tr - math.sqrt(disc)) / 2
slope = (l1 - a_) / c_
print(f"trace = {a_} + {b_} = {tr},   det = {a_}·{b_} − {c_}² = {det}")
print(f"λ² − {tr}λ + {det} = 0  ->  discriminant = {tr}² − 4·{det} = {disc},  √ = {math.sqrt(disc)}")
print(f"λ1 = {l1},  λ2 = {l2}    (check: λ1 + λ2 = {l1 + l2} = trace)")
print(f"eigenvector slope = (λ1 − a) / c = ({l1} − {a_}) / {c_} = {slope:.4f}  ->  angle = atan({slope:.4f}) = {math.degrees(math.atan(slope)):.2f}°")
print(f"fraction of spread kept by PC1 = {l1} / {tr} = {l1 / tr:.4f}")

### BUILD 7: `pca_2d`
Return `(angle, fraction)`: the PC1 angle in degrees, in the range [0, 180), and the fraction of spread it keeps. **Centre the points first.** Use the quadratic, not `np.linalg`.

<details><summary>Hint 1: angles</summary>

A direction and its opposite are the same line, so 200° and 20° describe the same component. `math.degrees(math.atan2(rise, run)) % 180` lands in [0, 180) and handles a negative slope correctly.

</details>

<details><summary>Hint 2: the special case</summary>

If c = 0, the formula (λ − a) ÷ c divides by zero. But then S is already **diagonal**: S·(1, 0) = (a, 0) and S·(0, 1) = (0, b), so the axes themselves are the eigenvectors. PC1 is the x-axis if a ≥ b, otherwise the y-axis.

</details>

In [ ]:
def pca_2d(points):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(pca_2d, [
    ((P,), (math.degrees(math.atan(1 / 3)), 11 / 12)),
    (([(1, 1), (-1, -1)],), (45.0, 1.0)),
    (([(1, 0), (-1, 0), (0, 0.5), (0, -0.5)],), (0.0, 0.8)),
    (([(0, 3), (0, -3), (1, 0), (-1, 0)],), (90.0, 0.9)),
    (([(1, -1), (-1, 1), (0.5, 0.5), (-0.5, -0.5)],), (135.0, 0.8)),
    (([(1, 2), (3, 4)],), (45.0, 1.0)),
])

### EXPLORE: check against numpy, then stretch the data
First, compare your `pca_2d` with numpy's eigenvalue routine on random clouds. Then a prediction:

**PREDICT:** double every y-value of our four points. Does the PC1 angle go up or down? Does PC1 keep more or less of the spread?

In [ ]:
if prep.ready(pca_2d, [(1, 0), (-1, 0), (0, 0.5), (0, -0.5)]):
    rng = np.random.default_rng(3)
    for t in range(3):
        cloud = rng.normal(size=(30, 2)) @ rng.normal(size=(2, 2))
        ang, frac = pca_2d([tuple(p) for p in cloud])
        cc = cloud - cloud.mean(axis=0)
        vals, vecs = np.linalg.eigh(cc.T @ cc)
        ang_np = math.degrees(math.atan2(vecs[1, -1], vecs[0, -1])) % 180
        print(f"cloud {t}: yours {ang:6.2f}°, {frac:.4f}    numpy {ang_np:6.2f}°, {vals[-1] / vals.sum():.4f}")
    for name, pts_ in [("original", P), ("y doubled", [(x, 2 * y) for x, y in P])]:
        ang, frac = pca_2d(pts_)
        print(f"{name:10s}: PC1 at {ang:.2f}°, keeps {frac:.3f} of the spread")
    ang, _ = pca_2d(P)
    plt.figure(figsize=(4, 4))
    plt.scatter(*zip(*P), zorder=3)
    r = np.array([-3, 3])
    plt.plot(r * math.cos(math.radians(ang)), r * math.sin(math.radians(ang)), label="PC1")
    plt.xlim(-3, 3); plt.ylim(-3, 3)
    plt.gca().set_aspect("equal"); plt.grid(alpha=.3); plt.legend(); plt.title("Our four points and PC1")
    plt.show()

Stretching y makes the vertical direction matter more, so PC1 tilts toward it and the cloud becomes less "one-dimensional". PCA depends on the units of your features. That's why people often rescale features before running it.

**Now do practice problem 3.5 on paper.** Estimate first: should the angle be above or below 45°? Then solve the quadratic.

In [ ]:
prep.check("3.5a", ...)   # angle in degrees (one of the options)
prep.check("3.5b", ...)   # floor(100 x fraction explained)

---
## 8. A linear classifier: s = Xw + b

**Idea.** Give each feature a **weight**. The **score** of an example is the weighted sum of its features plus a constant **bias** b. Predict class 1 if the score is **strictly greater than 0**, otherwise class 0. A score of exactly 0 is class 0. (That's the rule in the Jan 18 B4 section. Other papers may differ, so read it every time.)

With many examples, stack them as the rows of a matrix **X** (n rows = examples, d columns = features). The weights form a list **w** of d numbers. Each example's score is its row's dot product with w, plus b. Written for all rows at once: **s = Xw + b·1**, where **1** is a list of n ones (so b is added to every score).

**Smallest example.** Three yes/no gesture features: f1 = "shake detected", f2 = "fast", f3 = "upward". Weights w = (2, −1, 1), bias b = −1.

In [ ]:
Xs = [[1, 0, 0], [0, 1, 1], [1, 1, 0], [1, 1, 1]]
ws, bs_ = [2, -1, 1], -1
ys = [1, 0, 1, 1]
rows = []
for i, (row, yv) in enumerate(zip(Xs, ys), start=1):
    terms = " + ".join(f"{xv}·({wv})" for xv, wv in zip(row, ws))
    dot = sum(xv * wv for xv, wv in zip(row, ws))
    rows.append([i, row, terms, dot, dot + bs_, "?", yv])
prep.table(rows, header=["row", "x", "w·x terms", "w·x", "score = w·x + b", "prediction", "label"])

**PREDICT:** fill in the prediction column. What does the rule say about row 3, whose score is exactly 0? What is the accuracy (fraction of correct predictions)?

In [ ]:
s_np = np.array(Xs) @ np.array(ws) + bs_
pred_np = (s_np > 0).astype(int)
print("scores      s = Xw + b :", s_np)
print("predictions (s > 0)    :", pred_np)
print("labels                 :", np.array(ys))
print(f"accuracy = {(pred_np == np.array(ys)).sum()}/{len(ys)}")

Row 3 scores exactly 0, so it's predicted class 0, and that's wrong for its label. The strict ">" costs a point here. In numpy, `X @ w` is the matrix-times-vector product.

### BUILD 8: `predict_linear`
Return a list of 0/1 predictions, one per row of X. Remember: 1 only if the score is **> 0**.

In [ ]:
def predict_linear(X, w, b=0):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(predict_linear, [
    (([[1, 0], [0, 1], [1, 1]], [2, -2], 0), [1, 0, 0]),
    (([[1, 2]], [1, 1], -3), [0]),
    (([[1, 2]], [1, 1], -2.5), [1]),
    ((Xs, ws, bs_), [1, 0, 0, 1]),
])

### BUILD 9: `accuracy`
Return the fraction of predictions that equal the labels.

In [ ]:
def accuracy(pred, y):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(accuracy, [
    (([1, 0, 1], [1, 1, 1]), 2 / 3),
    (([0], [0]), 1.0),
    (([1, 0, 0, 1], ys), 0.75),
])

---
## 9. Reading weights off identity rows

**Idea.** Suppose you know some scores but not the weights. If a row of X is all zeros except a single 1 in column j, its score (with b = 0) is exactly w_j. **An identity row reads a weight straight off the score.** (It's called that because those rows look like rows of the **identity matrix**, which has 1s on the diagonal and 0s elsewhere.) A row with several 1s gives a *sum* of weights. Once all but one of those weights are known, a single subtraction gives the last one.

**Smallest example.** b = 0. Four rows, three unknown weights, and one score is missing (`None`).

In [ ]:
Xi = [[1, 0, 0], [0, 1, 0], [1, 1, 0], [0, 1, 1]]
si = [3, -2, None, 4]
prep.table([[i + 1, row, si[i] if si[i] is not None else "?"] for i, row in enumerate(Xi)], header=["row", "x", "score"])

**PREDICT:** which weights can you read with no arithmetic at all? Which row then gives w₃? What is the missing score in row 3?

In [ ]:
print("row 1 is an identity row (1 in column 1): w1 = s1 = 3")
print("row 2 is an identity row (1 in column 2): w2 = s2 = -2")
print("row 3 has an unknown score, so it can't help yet")
print("row 4: w2 + w3 = 4, so w3 = 4 - (-2) = 6")
print("now fill row 3: s3 = w1 + w2 = 3 + (-2) = 1")

The order matters: first read the identity rows, then use rows with exactly one unknown weight, then fill in unknown scores. This "peeling" works whenever each step leaves exactly one unknown.

### BUILD 10: `peel_weights`
`s` has `None` for unknown scores, and b = 0. Repeat until nothing changes: find a row with a known score where exactly one of its non-zero columns has an unknown weight, and solve for that weight. Return the list of weights, with `None` for any weight that can't be determined.

In [ ]:
def peel_weights(X, s):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(peel_weights, [
    ((Xi, si), [3, -2, 6]),
    (([[2, 0], [1, 1]], [4, 5]), [2, 3]),
    (([[1, 1]], [4]), [None, None]),
    (([[1, 1], [1, 0]], [None, 2]), [2, None]),
])

---
## 10. Pruning a feature, and sweeping the bias

**Idea.** **Pruning** feature j means setting w_j = 0: the model gets cheaper (fewer multiplications and less memory on the microcontroller). Each score drops by the **removed contribution** w_j · x_ij, where x_ij is row i's value in column j.
- Rows with x_ij = 0 don't change at all.
- If w_j is *already* 0, no score changes anywhere, so pruning that feature is **free**.

**Smallest example.** b = 0 throughout.

In [ ]:
Xp = [[1, 1, 0], [0, 1, 1], [1, 0, 1], [0, 1, 0]]
wp = [2, 0, -1]
yp = [1, 0, 1, 0]
base = [sum(a * b for a, b in zip(row, wp)) for row in Xp]
print("weights:", wp, "  full scores:", base, "  labels:", yp)

**PREDICT:** which feature can you prune without changing *any* score? Which one is the most expensive to prune? Is there a feature whose pruning changes scores but not the accuracy?

In [ ]:
for j in range(3):
    removed = [wp[j] * row[j] for row in Xp]
    new = [s - r for s, r in zip(base, removed)]
    pred = [1 if s > 0 else 0 for s in new]
    acc = sum(p == t for p, t in zip(pred, yp))
    print(f"prune feature {j + 1}: removed {removed} -> scores {new} -> predictions {pred} -> {acc}/4 correct")

Feature 2 has weight 0, so pruning it is free in every row. Feature 3 changes two scores but none of them crosses 0, so the accuracy survives. Feature 1 pushes two rows down onto or below 0, and both flip to wrong. Two ways to be cheap: a zero weight, or changes too small to cross the threshold.

### Sweeping the bias
After pruning, the scores may sit just on the wrong side of 0. Adding a bias b shifts *every* score by the same amount. Try each allowed b and count the correct predictions. If several b tie, the question will give a tie rule. Here: **pick the smallest |b|; if still tied, the more negative b.**

**PREDICT:** scores [1, −1, 0, −3, 2] with labels [1, 1, 1, 0, 1]. Which integer b from −3 to 3 is best? Careful with b = 3 and row 4.

In [ ]:
sc = [1, -1, 0, -3, 2]
ysc = [1, 1, 1, 0, 1]
rows = []
for b in range(-3, 4):
    shifted = [s + b for s in sc]
    pred = [1 if s > 0 else 0 for s in shifted]
    rows.append([b, shifted, pred, sum(p == t for p, t in zip(pred, ysc))])
prep.table(rows, header=["b", "scores + b", "predictions", "correct (of 5)"])

Two values tie for the best accuracy. At b = 3, row 4's score lands on exactly 0, which is still class 0 and so still correct. The tie rule picks the smaller |b|.

### BUILD 11: `best_bias`
Given scores, labels and a list of candidate biases, return the bias with the highest accuracy, breaking ties by the smallest |b|, then the more negative b.

In [ ]:
def best_bias(scores, y, bs):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(best_bias, [
    (([1, 2, 3], [0, 1, 1], list(range(-3, 4))), -1),
    ((sc, ysc, list(range(-3, 4))), 2),
    (([0, 0.5], [1, 0], [-1, 0, 1]), -1),
    (([5], [1], [-1, 0, 1]), 0),
])

---
## 11. Exam walkthrough: Jan 18, Section B4 (Model Compression)

> Do this **after** taking the Jan 18 paper as your diagnostic. Open `papers/2026-stage1-jan18/paper.pdf`, pages 23–26.

Read the two background boxes first (pages 23–24). Underline three things (move M1):
1. Predict 1 only if the score is **> 0**. A score of exactly 0 is class 0.
2. Indexing is **1-based**: feature 1 is the first column.
3. s = Xw + b·1, and in Q1 and Q2, b = 0.

### Q1: Reverse-engineering the weights (pages 24–25)

In [ ]:
X1 = [[1, 0, 0, 0],
      [0, 1, 0, 0],
      [0, 0, 1, 0],
      [0, 0, 0, 1],
      [1, 1, 0, 0],
      [1, 0, 1, 1]]
y1 = [1, 0, 1, 0, 1, 1]
sA = [2, None, 1, -1, 1, None]    # None = "?" on the paper
sB = [3, -5, None, -1, None, 1]

**Step 1.** Which rows of X are identity rows? For option A, which weights can you read straight off, which weight is still missing, and which row pins it down? Then the same for option B.

*Your answer here.*

<details><summary>Hint 1</summary>

Rows 1–4 are identity rows: row j says s_j = w_j. But each option hides one of those four scores behind a "?".

</details>

<details><summary>Hint 2</summary>

Option A hides s₂, so w₂ must come from another row that contains column 2 and has a known score. Option B hides s₃; which row contains column 3 and has a known score?

</details>

**Step 2.** Solve both weight vectors, then fill in the "?" scores, predict, and compute each option's accuracy. Enter your weights below. The checker only tells you whether X·w reproduces every *known* score of that option.

In [ ]:
wA = [..., ..., ..., ...]
wB = [..., ..., ..., ...]

def check_q1(wA, wB):
    for name, w, s in [("A", wA, sA), ("B", wB, sB)]:
        if any(v is ... for v in w):
            print(f"option {name}: fill in all four weights first."); continue
        bad = [i + 1 for i, (row, si) in enumerate(zip(X1, s))
               if si is not None and not math.isclose(sum(a * b for a, b in zip(row, w)), si, abs_tol=1e-9)]
        print(f"option {name}: " + ("consistent with all 4 known scores" if not bad else f"does NOT reproduce the known score in row(s) {bad}"))

check_q1(wA, wB)

<details><summary>Hint 3</summary>

The "?" scores aren't free choices. Once w is known, every score is fixed by s = Xw. A filled-in "?" can come out on the wrong side of 0 for its label, and that's what lowers an option's accuracy.

</details>

**Step 3.** The **ℓ1 norm** (say "L-one") of a vector is the sum of the absolute values of its entries: ‖w‖₁ = |w₁| + |w₂| + … For example, ‖(1, −2)‖₁ = 3. Compute it for the weights of the option with the higher accuracy.

In [ ]:
prep.check("J18-B4-Q1", ...)   # floor of the l1 norm

### Q2: The best pair of features to prune (page 25)

In [ ]:
X2 = [[-1,  0,  0,  0,  0, 0,  3],
      [ 0, -1,  0,  0,  0, 0,  0],
      [ 0,  0, -1,  0,  0, 0, -2],
      [ 0,  0,  0, -1,  0, 0,  6],
      [ 0,  0,  0,  0, -1, 0,  7],
      [ 0,  0,  0,  0,  0, 1, -6],
      [ 1,  1,  1,  1,  1, 5, -9]]
w2 = [-1, -1, -1, -1, -1, 1, 0]
y2 = [1, 1, 1, 1, 1, 1, 0]

**Step 1.** Before pruning anything, compute all 7 scores s = Xw and the accuracy. Enter your scores to check them.

In [ ]:
my_s2 = [..., ..., ..., ..., ..., ..., ...]

def check_scores(s):
    if any(v is ... for v in s):
        print("Fill in all 7 scores first."); return
    for i, (row, v) in enumerate(zip(X2, s), start=1):
        ok = math.isclose(sum(a * b for a, b in zip(row, w2)), v, abs_tol=1e-9)
        print(f"row {i}: {'ok' if ok else 'CHECK'}")

check_scores(my_s2)

**Step 2.** Look at w. One weight is already 0. What does pruning that feature do to every score?

<details><summary>Hint</summary>

The removed contribution is r_i = w_j · x_ij. If w_j = 0, then r_i = 0 in every row, so pruning feature j is free (section 10).

</details>

**Step 3.** You must prune **exactly two** features. Any pair that includes the free feature scores exactly the same as pruning the other feature on its own. So a good first question is: **which single feature hurts least to prune?** But is the best pair guaranteed to include the free one? Not automatically. A second prune might repair a row that the first prune broke, or a weight might have been pushing a score the wrong way all along. Compare with your baseline accuracy from Step 1, and check whether any pair of two non-free features could do better.

<details><summary>Hint 1</summary>

Pruning feature j changes only the rows where x_ij ≠ 0. For each feature, list the rows it touches.

</details>

<details><summary>Hint 2</summary>

For each candidate, write the new scores of just the touched rows and count how many land on the wrong side of 0. Remember that exactly 0 means class 0.

</details>

Try it on paper first. Then, if you want, check every one of the 21 pairs in code with your `predict_linear` and `accuracy`:

In [ ]:
# Your attempt: loop over all pairs {j, k} (itertools.combinations), prune both, and compute the accuracy.

In [ ]:
prep.check("J18-B4-Q2", ...)   # j* x k*

### Q3: Sweeping the bias (page 26)
Take your best pair from Q2 and its pruned scores. Add an integer b ∈ {−3, …, 3} to every score, compute the accuracy for each b, and apply the paper's tie rule: the smallest |b|, then the more negative b. Report j*·k* + |b*|.

The background box also describes **bias correction**, b′ = b + μ_S. Q3 doesn't ask for it. It asks you to sweep integer b. Answer the question that's asked (move M1).

<details><summary>Hint 1</summary>

After pruning, which rows are wrong, and what are their scores? Which way must b move them?

</details>

<details><summary>Hint 2</summary>

Raising b fixes rows with label 1 and a score ≤ 0, but it can break a row with label 0 if that row's score climbs above 0. How far can b go before that happens?

</details>

<details><summary>Hint 3</summary>

Several values of b may tie for the best accuracy. Apply the tie rule exactly as written.

</details>

In [ ]:
prep.check("J18-B4-Q3", ...)   # j* x k* + |b*|

---
## Back to the hook

You now know five ways your robot could classify or group its 20 gestures. For each one, write one line: what would the microcontroller have to *store*, and where could a tie or a threshold trip it up?

*Your answer here.*

---
## 12. Connect

Answer in your own words in the cell below. Then copy the best sentence into `log/ERROR_LOG.md` under "Moves I own".

1. k-means moves each centroid to the **mean** of its points, because the mean minimises the sum of squared distances (notebook 01, problem 1.4). If you changed the goal to the sum of *absolute* distances, what should the centroid step use instead?
2. k-means repeats assign → average until nothing changes. The Markov chain in notebook 02 repeats a step until the distribution stops changing. Both stop at a **fixed point**. Why can k-means end at different fixed points from different starts, while the Wi-Fi chain always reached the same one?
3. A PCA shadow is t = x · cos θ + y · sin θ. A linear classifier's score is s = Xw + b. Both are the same operation: a dot product with a vector. What would "thresholding the PC1 shadow at 0" do to the 20 gestures, and why might it, or might it not, separate waves from punches?

*Your answers here.*